# ML&F - Scikit-learn ML Pipelines

## Step 0: Load the dataset


We need to define our data and target. In this case we build a classification model

In [1]:
import pandas as pd

ames_housing = pd.read_csv("datasets/house_prices.csv", na_values="?")
ames_housing['LotArea']

0        8450
1        9600
2       11250
3        9550
4       14260
        ...  
1455     7917
1456    13175
1457     9042
1458     9717
1459     9937
Name: LotArea, Length: 1460, dtype: int64

In [2]:
target_name = "SalePrice"
data, target = (
    ames_housing.drop(columns=target_name), # remove what we want to predict.
    ames_housing[target_name],
)
target = (target > 200_000).astype(int)

We inspect the first rows of the dataframe



In [3]:
data.head(10)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal
5,6,50,RL,85.0,14115,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,Shed,700,10,2009,WD,Normal
6,7,20,RL,75.0,10084,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,8,2007,WD,Normal
7,8,60,RL,NaN,10382,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Shed,350,11,2009,WD,Normal
8,9,50,RM,51.0,6120,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2008,WD,Abnorml
9,10,190,RL,50.0,7420,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,1,2008,WD,Normal


In [4]:
target.head(10)

0    1
1    0
2    1
3    0
4    1
5    0
6    1
7    0
8    0
9    0
Name: SalePrice, dtype: int64

For the sake of simplicity, we can cherry-pick some features and only retain this arbitrary subset of data:



In [5]:
numeric_features = ["LotArea", "FullBath", "HalfBath"]
categorical_features = ["Neighborhood", "HouseStyle"]
data = data[numeric_features + categorical_features]
data.head()

,LotArea,FullBath,HalfBath,Neighborhood,HouseStyle
0,8450,2,1,CollgCr,2Story
1,9600,2,0,Veenker,1Story
2,11250,2,1,CollgCr,2Story
3,9550,1,0,Crawfor,2Story
4,14260,2,1,NoRidge,2Story


## Step 1: Train-Test Split (Critical First Step!)

**Before any data preprocessing or pipeline creation**, we must split our data into training and testing sets. This prevents data leakage by ensuring that:
- No information from the test set influences our preprocessing steps
- Our model evaluation is unbiased and realistic
- We follow proper ML workflow practices

In [6]:
from sklearn.model_selection import train_test_split

# Split data BEFORE any preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    data, target, 
    test_size=0.2, 
    random_state=42, 
    stratify=target  # Ensures balanced splits for classification
)

## Step 2: Create the ML Pipeline for Preprocessing

Now we create our pipeline. The key advantage is that all preprocessing steps will be applied consistently to both training and test data, and the pipeline learns preprocessing parameters **only from the training data**.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",StandardScaler()),
    ]
)

categorical_transformer = OneHotEncoder(handle_unknown="ignore")

The next step is to apply the transformations using ``ColumnTransformer``



In [8]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

Then we define the model and join the steps in order



**Important**: We fit the pipeline only on training data. This ensures that:
- Imputation values (median) are calculated only from training data
- Scaling parameters (mean, std) are calculated only from training data  
- One-hot encoding learns categories only from training data
- No information leaks from the test set

In [ ]:
from sklearn.linear_model import LogisticRegression

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor), # the processor is in a pipeline and the model training is also in a pipeline.
        ("classifier", LogisticRegression()),
    ]
)
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3I

Now we can evaluate our trained pipeline on the test set to get an unbiased performance estimate:

### My notes: one house through Step 2

Step 2 only writes the recipe. The numbers below are learned later, at `model.fit(X_train, y_train)` in Step 3, from the 1168 training houses.

Scaling formula: **scaled = (value - mean) / std**

- mean = the average house
- std = how far a typical house is from that average (one "step")
- after scaling, every number column has mean 0 and std 1. One mean and one std per column, shared by all rows.

```mermaid
flowchart TD
    H[One house<br/>9825, 2, 0<br/>Edwards, 1Story]:::root --> CT[ColumnTransformer<br/>sorts the columns]:::branch
    CT --> NUM[num: 3 columns<br/>median imputer<br/>then scaler]:::branch
    CT --> CAT[cat: 2 columns<br/>OneHotEncoder]:::branch
    NUM --> L[LotArea<br/>9825 - 10719<br/>/ 10808 = -0.08]
    NUM --> F[FullBath<br/>2 - 1.56<br/>/ 0.55 = +0.79]
    NUM --> HB[HalfBath<br/>0 - 0.38<br/>/ 0.50 = -0.76]
    CAT --> N[Neighborhood<br/>25 columns<br/>Edwards = 1]
    CAT --> S[HouseStyle<br/>8 columns<br/>1Story = 1]
    L --> OUT[36 columns<br/>3 + 25 + 8]:::root
    F --> OUT
    HB --> OUT
    N --> OUT
    S --> OUT
    OUT --> LR[LogisticRegression]:::branch

    classDef root fill:#9fc56f,stroke:#555,color:#111
    classDef branch fill:#b9d79a,stroke:#555,color:#111
    classDef default fill:#dde9d0,stroke:#555,color:#111
```

| column | house | mean (train) | std (train) | scaled | read as |
|---|---|---|---|---|---|
| LotArea | 9825 | 10,719 | 10,808 | -0.08 | almost average |
| FullBath | 2 | 1.56 | 0.55 | +0.79 | 0.8 steps above average |
| HalfBath | 0 | 0.38 | 0.50 | -0.76 | 0.8 steps below average |

FullBath in train: 6 houses with 0, 529 with 1, 604 with 2, 29 with 3. That's why the average is 1.56.

Before scaling, LotArea (9825) is about 5000 times bigger than FullBath (2) only because it's in square feet. After scaling both are in "steps from average".

The imputer does nothing here: these 5 columns have no missing values. It would fill an empty LotArea with the train median, 9525.

Leakage link: fitting the scaler on all 1460 houses moves the LotArea mean to 10,517 and the std to 9,978, because the 292 test houses get a say.

Still to check: a test house with 4 full baths (train never had one). What does the scaler give it?

## Step 3: Train the Pipeline

In [10]:
model.fit(X_train, y_train)  # Fit ONLY on training data

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['LotArea','FullBath','HalfBath','Neighborhood','HouseStyle']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaini

## Step 4: Evaluate on Test Set

### Option 1: Simple Train-Test Evaluation

In [11]:
test_score = model.score(X_test, y_test)
train_score = model.score(X_train, y_train)

print(f"Training accuracy: {train_score:.3f}")
print(f"Test accuracy: {test_score:.3f}")
print(f"Difference: {abs(train_score - test_score):.3f}")

Training accuracy: 0.874
Test accuracy: 0.866
Difference: 0.008


### Option 2: Cross-Validation on Training Set

In [12]:
# We can also do cross-validation, but ONLY on training data
from sklearn.model_selection import cross_validate

cv_results = cross_validate(model, X_train, y_train, cv=5)
scores = cv_results["test_score"]
print(
    "The mean cross-validation accuracy is: "
    f"{scores.mean():.3f} ± {scores.std():.3f}"
)

The mean cross-validation accuracy is: 0.860 ± 0.022


In [13]:
# Get predictions on test set
test_score = model.score(X_test, y_test)
train_score = model.score(X_train, y_train)

print(f"Training accuracy: {train_score:.3f}")
print(f"Test accuracy: {test_score:.3f}")
print(f"Difference: {abs(train_score - test_score):.3f}")

# This shows if we have overfitting
if train_score - test_score > 0.05:
    print("⚠️  Potential overfitting detected!")
else:
    print("✅ Good generalization performance")

Training accuracy: 0.874
Test accuracy: 0.866
Difference: 0.008
✅ Good generalization performance


Notice that the diagram changes color once the estimator is fit.

So far we used ``Pipeline`` and ``ColumnTransformer``, which allows us to custom the names of the steps in the pipeline. An alternative is to use ``make_column_transformer`` and ``make_pipeline``, they do not require, and do not permit, naming the estimators. Instead, their names are set to the lowercase of their types automatically.

In [14]:
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

numeric_transformer = make_pipeline(
    SimpleImputer(strategy="median"), StandardScaler()
)
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = make_column_transformer(
    (numeric_transformer, numeric_features),
    (categorical_transformer, categorical_features),
)
model = make_pipeline(preprocessor, LogisticRegression())
model.fit(X_train, y_train)  # Fit ONLY on training data

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['LotArea','FullBath','HalfBath','Neighborhood','HouseStyle']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('pipeline', ...), ('onehotencoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder=

## Key Takeaways


**✅ What we did RIGHT:**
1. **Split data FIRST** before any preprocessing or analysis
2. **Fit pipeline only on training data** - all preprocessing parameters learned from training set only
3. **Evaluate on test set** for unbiased performance estimate
4. **Use cross-validation on training data** for robust model selection

**❌ Common mistakes that cause data leakage:**
- Preprocessing entire dataset before train-test split
- Using statistics from test set for normalization/scaling  
- Feature selection using entire dataset
- Cross-validating on entire dataset including test set

**🔑 Why pipelines help:**
- Ensure consistent preprocessing between training and inference
- Automatically apply same transformations to new data
- Make it impossible to accidentally use test set information during training

Note:
In this case, around 86% of the times the pipeline correctly predicts whether the price of a house is above or below the 200_000 dollars threshold. But be aware that this score was obtained by picking some features by hand, which is not necessarily the best thing we can do for this classification task. In this example we can hope that fitting a complex machine learning pipelines on a richer set of features can improve upon this performance level.

Reducing a price estimation problem to a binary classification problem with a single threshold at 200_000 dollars is probably too coarse to be useful in in practice. Treating this problem as a regression problem is probably a better idea. 

### My notes: reading the three scores

| score | data | my result |
|---|---|---|
| train accuracy | X_train, the rows the model learned from | 0.874 |
| CV accuracy | X_train cut into 5 folds, each fold tested once | 0.860 ± 0.022 |
| test accuracy | X_test, locked until the end | 0.866 |

The three numbers agree, so there's no sign of overfitting or leakage here.

- Train much higher than CV and test (say 0.99 vs 0.86): overfitting. The model memorized the training rows.
- CV or test suspiciously high: suspect leakage. In the exercise, `retention_offer_sent` alone gets a CV AUC of 0.956, while the honest features get 0.703.

`cross_validate` gets `X_train`, never `data`. `data` still holds the test rows, and they'd end up in the training folds.